In [ ]:
# Cargar dataset del restaurante
import pandas as pd

# Cargar dataset simulado
df = pd.read_csv("/content/dataset_simulado_opiniones.csv")

df.head()


,fecha,sentimiento,aspecto,comentario,calificacion
0,2025-11-08,positivo,tiempo de espera,"La tiempo de espera estuvo excelente, definiti...",4
1,2025-11-05,positivo,ambiente,"Muy buena ambiente, vale totalmente la pena. (...",4
2,2025-11-05,neutral,tiempo de espera,"La tiempo de espera estuvo bien, nada fuera de...",3
3,2025-11-05,negativo,comida,"La comida dejó mucho que desear, no volvería. ...",2
4,2025-11-14,positivo,comida,"La comida estuvo excelente, definitivamente vo...",5


In [ ]:
# Preparar datos para inferencia
textos = df["comentario"].tolist()
labels_reales = df["sentimiento"].tolist()

# Cargar modelo de análisis de sentimientos
from transformers import pipeline

clf_es = pipeline(
    task="text-classification",
    model="pysentimiento/robertuito-sentiment-analysis",
    device=0
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda:0


In [ ]:
# Inferencia y medición de tiempo
import time

inicio = time.time()
predicciones = clf_es(textos)
fin = time.time()

tiempo_inferencia = fin - inicio
tiempo_promedio = tiempo_inferencia / len(textos)

tiempo_inferencia, tiempo_promedio


(2.608435869216919, 0.010433743476867676)

In [ ]:
# Normalizar etiquetas predichas
labels_predichos = [p["label"] for p in predicciones]

# Se alinean las métricas de ambos vectores (labels_reales_norm y labels_predichos)
mapeo_labels = {
    "negativo": "NEG",
    "neutral": "NEU",
    "positivo": "POS"
}

labels_reales_norm = [
    mapeo_labels[l.strip().lower()] for l in labels_reales
]

set(labels_reales_norm), set(labels_predichos)


({'NEG', 'NEU', 'POS'}, {'NEG', 'NEU', 'POS'})

In [ ]:
# Cálculo de métricas (Accuracy, Precision, Recall, F1)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

accuracy = accuracy_score(labels_reales_norm, labels_predichos)

precision, recall, f1, _ = precision_recall_fscore_support(
    labels_reales_norm,
    labels_predichos,
    average="weighted"
)

# Despliegue de métricas y medición de tiempo
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")
print(f"Tiempo total inferencia(s): {tiempo_inferencia:.2f}")
print(f"Tiempo promedio por reseña(s): {tiempo_promedio:.4f}")

# Decisión automática: ¿Se requiere fine-tuning?
requiere_finetuning = (accuracy < 0.80) or (f1 < 0.80)

print(f"\n¿Se requiere fine-tuning?: {requiere_finetuning}")

Accuracy: 0.77
Precision: 0.83
Recall: 0.77
F1-score: 0.68
Tiempo total inferencia(s): 2.61
Tiempo promedio por reseña(s): 0.0104

¿Se requiere fine-tuning?: True


In [ ]:
# Fine-tuning (solo si es necesario - Parte 1 de 2)

# Se columna de etiquetas normalizadas en el dataset
mapeo_labels = {
    "negativo": "NEG",
    "neutral": "NEU",
    "positivo": "POS"
}

df["label_norm"] = (
    df["sentimiento"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(mapeo_labels)
)

# Verifica que se haya creado la columna de etiquetas
df["label_norm"].value_counts()



,count
label_norm,
POS,144
NEU,61
NEG,45


In [ ]:
# Fine-tuning (solo si es necesario - Parte 2 de 2)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset

# Tokenizador
tokenizer = AutoTokenizer.from_pretrained(
    "pysentimiento/robertuito-sentiment-analysis"
)

def tokenize(batch):
    return tokenizer(batch["comentario"], truncation=True, padding=True)

# Preparar labels numéricos
df["labels"] = df["label_norm"].map({
    "NEG": 0,
    "NEU": 1,
    "POS": 2
})

dataset = Dataset.from_pandas(df)
dataset = dataset.map(tokenize, batched=True)

# Modelo
model = AutoModelForSequenceClassification.from_pretrained(
    "pysentimiento/robertuito-sentiment-analysis",
    num_labels=3
)

# Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./resultados_restaurante",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()


Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Step,Training Loss


TrainOutput(global_step=96, training_loss=0.020581369598706562, metrics={'train_runtime': 17.6291, 'train_samples_per_second': 42.543, 'train_steps_per_second': 5.446, 'total_flos': 8093820955500.0, 'train_loss': 0.020581369598706562, 'epoch': 3.0})